# Jensen's Inequality

## Learning Objectives
- Define **convex** and **concave** functions via the second-derivative test and the chord condition
- State **Jensen's inequality**: for a convex $f$, $\; E[f(X)] \geq f(E[X])$
- Identify when equality holds (constant random variable)
- Reverse the inequality for concave $f$ — the key case used in EM (since $\log$ is concave)
- Understand why Jensen's inequality is the engine behind the EM lower bound

## Problem Statement

**Convexity:** A function $f : \mathbb{R} \to \mathbb{R}$ is **convex** if for all $x$, $f''(x) \geq 0$.
(For vector-valued inputs, the Hessian $\nabla^2 f \succeq 0$.)
It is **strictly convex** if $f''(x) > 0$ for all $x$.

Equivalently, $f$ is convex if the chord between any two points lies **above** the function:
$$f(\lambda x + (1-\lambda) y) \leq \lambda f(x) + (1-\lambda) f(y) \quad \forall\, x, y,\; \lambda \in [0,1]$$

**Jensen's Inequality (convex case):** For any random variable $X$ and convex $f$:
$$\boxed{E[f(X)] \geq f(E[X])}$$

Equality holds if and only if $X$ is a constant (i.e., $X = E[X]$ with probability 1).

**Jensen's Inequality (concave case):** If $f$ is concave ($f''(x) \leq 0$), the inequality reverses:
$$E[f(X)] \leq f(E[X])$$

**Why this matters for EM:** The log function is concave. Applying Jensen's to $\log$ allows us to move the $\log$ outside a sum, converting an intractable $\log \sum$ into a tractable $\sum \log$.

## 1. Convex vs. Concave: Geometric Picture

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0.1, 4, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

configs = [
    ('Convex: $f(x) = x^2$',   lambda x: x**2,   'Chord lies ABOVE curve\n$E[f(X)] \\geq f(E[X])$', '#2166ac'),
    ('Concave: $f(x) = \\log x$', lambda x: np.log(x), 'Chord lies BELOW curve\n$E[f(X)] \\leq f(E[X])$', '#d6604d'),
    ('Linear: $f(x) = 2x$',    lambda x: 2*x,    'Chord IS the curve\n$E[f(X)] = f(E[X])$', '#4dac26'),
]

x1, x2 = 1.0, 3.5
lam = 0.35
x_bar = lam * x1 + (1 - lam) * x2

for ax, (title, fn, annotation, color) in zip(axes, configs):
    ax.plot(x, fn(x), color=color, lw=2.5, label='$f(x)$')

    # Chord from (x1, f(x1)) to (x2, f(x2))
    chord_y = lam * fn(x1) + (1 - lam) * fn(x2)
    ax.plot([x1, x2], [fn(x1), fn(x2)], 'k--', lw=2, label='Chord')
    ax.scatter([x1, x2], [fn(x1), fn(x2)], s=80, c='k', zorder=5)

    # Key points: chord value vs f(x_bar)
    ax.scatter(x_bar, chord_y, s=150, c='orange', zorder=6, label=f'Chord at $\\bar{{x}}$: {chord_y:.2f}')
    ax.scatter(x_bar, fn(x_bar), s=150, c=color,   zorder=6, label=f'$f(\\bar{{x}})$: {fn(x_bar):.2f}', marker='D')

    # Vertical arrow showing gap
    gap = chord_y - fn(x_bar)
    if abs(gap) > 0.01:
        ax.annotate('', xy=(x_bar, fn(x_bar)), xytext=(x_bar, chord_y),
                    arrowprops=dict(arrowstyle='<->', color='purple', lw=2))

    ax.set_title(title, fontsize=10)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$f(x)$')
    ax.legend(fontsize=8)
    ax.text(0.03, 0.97, annotation, transform=ax.transAxes, fontsize=8.5,
            va='top', bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle("Jensen's Inequality — Convex, Concave, and Linear Functions",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Empirical Verification with Discrete Distributions

Jensen's inequality holds for any random variable $X$ with any distribution.
We verify both the convex and concave cases numerically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Discrete distribution: X takes values x_vals with probabilities p_vals
x_vals = np.array([0.5, 1.0, 1.5, 2.5, 3.5, 4.0])
p_vals = np.array([0.1, 0.25, 0.15, 0.3, 0.12, 0.08])
p_vals /= p_vals.sum()

EX = np.sum(p_vals * x_vals)

cases = [
    ('Convex: $f(x) = x^2$',    lambda x: x**2,    '$E[f(X)] \\geq f(E[X])$  ✓', '#2166ac'),
    ('Concave: $f(x) = \\log x$', lambda x: np.log(x), '$E[f(X)] \\leq f(E[X])$  ✓', '#d6604d'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

x_cont = np.linspace(0.1, 4.5, 300)

for ax, (title, fn, direction, color) in zip(axes, cases):
    # Compute Jensen quantities
    Efx  = np.sum(p_vals * fn(x_vals))   # E[f(X)]
    fEx  = fn(EX)                          # f(E[X])

    ax.plot(x_cont, fn(x_cont), color=color, lw=2.5, label='$f(x)$')

    # Stem plot: distribution
    ax.vlines(x_vals, 0, fn(x_vals), color='gray', lw=1.5, alpha=0.5)
    ax.scatter(x_vals, fn(x_vals), s=60, c='gray', zorder=5, alpha=0.7)

    # Mark E[f(X)] and f(E[X])
    ax.axhline(Efx, color='orange', ls='--', lw=2, label=f'$E[f(X)] = {Efx:.3f}$')
    ax.scatter(EX, fEx, s=200, c='purple', zorder=7, marker='*',
               label=f'$f(E[X]) = {fEx:.3f}$')
    ax.axvline(EX, color='purple', ls=':', lw=1.5, alpha=0.7)

    gap = Efx - fEx
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$f(x)$')
    ax.legend(fontsize=8.5)
    ax.text(0.03, 0.03, f'{direction}\nGap = {gap:+.4f}',
            transform=ax.transAxes, fontsize=9, va='bottom',
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle("Jensen's Inequality — Empirical Verification (discrete distribution)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Verification:')
for title, fn, direction, _ in cases:
    Efx = np.sum(p_vals * fn(x_vals))
    fEx = fn(EX)
    print(f'  {title[:20]}: E[f(X)]={Efx:.4f}, f(E[X])={fEx:.4f}, gap={Efx-fEx:+.4f}')

## 3. The Log Function and Why It Matters for EM

The log function is **strictly concave** ($\log''(x) = -1/x^2 < 0$), so Jensen gives:
$$E[\log X] \leq \log E[X]$$

Equivalently, for weights $Q_i(z) \geq 0$ summing to 1:
$$\sum_z Q(z) \log\left(\frac{p(x, z)}{Q(z)}\right) \leq \log\left(\sum_z Q(z) \cdot \frac{p(x, z)}{Q(z)}\right) = \log p(x)$$

This is **the key step** in deriving the EM lower bound: Jensen converts the intractable $\log \sum_z p(x,z)$ into the tractable $\sum_z Q(z) \log \frac{p(x,z)}{Q(z)}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

x = np.linspace(0.05, 5, 300)

# Left: concavity of log with Jensen illustration
ax = axes[0]
ax.plot(x, np.log(x), '#d6604d', lw=2.5, label='$f(x) = \\log x$ (concave)')

# Four sample points from a distribution over x
xs = np.array([0.5, 1.5, 2.5, 4.0])
qs = np.array([0.2, 0.4, 0.25, 0.15])
Ex = np.sum(qs * xs)
E_log_x = np.sum(qs * np.log(xs))
log_Ex   = np.log(Ex)

for xi, qi in zip(xs, qs):
    ax.plot([xi, xi], [0, np.log(xi)], 'gray', lw=1, ls=':')
    ax.scatter(xi, np.log(xi), s=50*qi*10, c='gray', zorder=5)

ax.axhline(E_log_x, color='orange', ls='--', lw=2, label=f'$E[\\log X] = {E_log_x:.3f}$')
ax.scatter(Ex, log_Ex, s=200, c='purple', marker='*', zorder=7,
           label=f'$\\log E[X] = {log_Ex:.3f}$')
ax.axvline(Ex, color='purple', ls=':', lw=1.5, alpha=0.7)

ax.set_xlabel('$x$')
ax.set_ylabel('$\\log x$')
ax.set_title('$\\log$ is concave:\n$E[\\log X] \\leq \\log E[X]$')
ax.legend(fontsize=9)
ax.text(0.03, 0.03, f'Gap = {E_log_x - log_Ex:+.4f}\n(always ≤ 0 for concave f)',
        transform=ax.transAxes, fontsize=9, va='bottom',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

# Right: gap size as a function of variance (spread) of X
ax = axes[1]
rng = np.random.default_rng(0)
mean_val = 2.0
variances = np.linspace(0.01, 1.5, 40)
gaps = []
for var in variances:
    samples = rng.lognormal(np.log(mean_val) - var/2, np.sqrt(var), 10000)
    gaps.append(np.log(np.mean(samples)) - np.mean(np.log(samples)))

ax.plot(variances, gaps, 'b-o', lw=2.5, ms=5)
ax.axhline(0, color='k', ls='--', lw=1)
ax.set_xlabel('Variance of $X$')
ax.set_ylabel('$\\log E[X] - E[\\log X]$ (Jensen gap)')
ax.set_title('Jensen gap grows with variance\n(zero iff $X$ is constant)')
ax.text(0.03, 0.97,
        'Equality in Jensen\'s holds\nonly when $X$ is constant\n(zero variance)',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

fig.suptitle("Jensen's Inequality for $\\log$ — Foundation of the EM Lower Bound",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Jensen's Inequality Gap and the Equality Condition

The gap $E[f(X)] - f(E[X]) \geq 0$ (for convex $f$) measures how "spread out" $X$ is.

**Equality condition:** For strictly convex (or strictly concave) $f$, equality holds
if and only if $X$ is **constant** — i.e., $P(X = E[X]) = 1$.

In the EM context: the lower bound $J(Q, \theta)$ equals $\ell(\theta)$ exactly when
$Q_i(z) = p(z \mid x^{(i)}; \theta)$ — the posterior distribution — making the ratio
$\frac{p(x^{(i)}, z; \theta)}{Q_i(z)}$ constant in $z$. This is the **E-step choice**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Illustrate: lower bound J(Q, theta) vs log-likelihood ell(theta)
# as Q moves from uniform to the posterior (tight bound)

rng = np.random.default_rng(7)

# Simple 1D GMM: k=2, fixed data point x0
# p(x, z=1) = phi1 * N(x; mu1, sig1)
# p(x, z=2) = phi2 * N(x; mu2, sig2)
from scipy.stats import norm

phi1, phi2 = 0.4, 0.6
mu1, mu2   = -1.5, 1.5
s1, s2     = 1.0, 1.0
x0         = 1.0   # a single observed point

p_x_z1 = phi1 * norm.pdf(x0, mu1, s1)
p_x_z2 = phi2 * norm.pdf(x0, mu2, s2)
px     = p_x_z1 + p_x_z2
log_px = np.log(px)

# Posterior (tight Q)
Q_post = np.array([p_x_z1 / px, p_x_z2 / px])

# Vary Q1 from 0 to 1 (Q2 = 1 - Q1)
q1_vals = np.linspace(0.01, 0.99, 200)
J_vals  = []
for q1 in q1_vals:
    q2 = 1 - q1
    Q  = np.array([q1, q2])
    p  = np.array([p_x_z1, p_x_z2])
    # J = sum_z Q(z) * log(p(x,z)/Q(z))
    J  = q1 * np.log(p_x_z1 / q1 + 1e-300) + q2 * np.log(p_x_z2 / q2 + 1e-300)
    J_vals.append(J)

J_at_posterior = (Q_post[0] * np.log(p_x_z1 / Q_post[0]) +
                  Q_post[1] * np.log(p_x_z2 / Q_post[1]))

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(q1_vals, J_vals, 'b-', lw=2.5, label='Lower bound $J(Q, \\theta)$')
ax.axhline(log_px, color='green', ls='--', lw=2, label=f'$\\ell(\\theta) = \\log p(x) = {log_px:.3f}$')
ax.axvline(Q_post[0], color='red', ls=':', lw=2,
           label=f'Posterior $Q_1 = p(z=1|x) = {Q_post[0]:.3f}$')
ax.scatter(Q_post[0], J_at_posterior, s=200, c='red', zorder=7, marker='*',
           label=f'Tight: $J = \\ell$ at posterior ({J_at_posterior:.3f})')

ax.set_xlabel('$Q(z=1)$ (weight on first component)')
ax.set_ylabel('$J(Q, \\theta)$ or $\\ell(\\theta)$')
ax.set_title('Jensen lower bound $J(Q,\\theta) \\leq \\ell(\\theta)$\n'
             'Equality (tight bound) at $Q_i = p(z|x;\\theta)$ — the E-step choice')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'log p(x) = {log_px:.4f}')
print(f'J at posterior Q = {J_at_posterior:.4f}  (difference: {log_px - J_at_posterior:.2e})')
print(f'Posterior: Q(z=1) = {Q_post[0]:.4f}, Q(z=2) = {Q_post[1]:.4f}')

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="374"
     viewBox="0 0 780 374" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: Convex/concave f -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Convex f: f&#x2033; &#x2265; 0</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Concave f: f&#x2033; &#x2264; 0</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >chord lies above (convex) or below (concave) the function</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="82" font-size="11.5" font-style="italic" fill="#555"
        >apply to a random variable X</text>

  <!-- Row 2: Jensen's inequality -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Jensen&#x2019;s</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">inequality</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >convex: E[f(X)] &#x2265; f(E[X]); concave: E[f(X)] &#x2264; f(E[X])</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="182" font-size="11.5" font-style="italic" fill="#555"
        >apply to log (concave): &#x2211; Q(z) log[p(x,z)/Q(z)] &#x2264; log p(x)</text>

  <!-- Row 3: EM lower bound -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="240" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">EM lower bound</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >J(Q,&#x3b8;) = &#x2211;&#x2071; &#x2211;&#x2c7; Q&#x2071;(z) log[p(x&#x2071;,z;&#x3b8;)/Q&#x2071;(z)] &#x2264; &#x2113;(&#x3b8;)</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="282" font-size="11.5" font-style="italic" fill="#555"
        >tight when Q&#x2071;(z) = p(z|x&#x2071;;&#x3b8;) (equality in Jensen)</text>

  <!-- Row 4: E-step -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="340" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">E-step choice</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >set Q&#x2071;(z) = posterior &#x2014; makes J(Q,&#x3b8;) = &#x2113;(&#x3b8;) at current &#x3b8; (tight lower bound)</text>
</svg>
""")

## Summary

| Concept | Statement | Key Insight |
|---|---|---|
| Convexity | $f''(x) \geq 0$ everywhere | Chord lies above graph |
| Concavity | $f''(x) \leq 0$ everywhere | Chord lies below graph |
| Jensen (convex) | $E[f(X)] \geq f(E[X])$ | Expectation outside $\leq$ inside |
| Jensen (concave) | $E[f(X)] \leq f(E[X])$ | Expectation outside $\geq$ inside |
| Equality condition | $X$ is constant ($P(X=E[X])=1$) | Larger variance → larger gap |
| $\log$ is concave | $(\log x)'' = -1/x^2 < 0$ | Key to EM lower bound derivation |
| EM lower bound | $J(Q,\theta) = \sum_{i,z} Q_i(z)\log\frac{p(x^{(i)},z;\theta)}{Q_i(z)} \leq \ell(\theta)$ | Jensen applied to $\log$ |
| Tight bound | $Q_i(z) = p(z\lvert x^{(i)};\theta)$ | E-step choice that tightens the bound |

**Key insight:** Jensen's inequality applied to the concave $\log$ function gives a tractable lower bound on the intractable marginal log-likelihood; the E-step makes this bound tight by setting $Q$ to the posterior — the entire EM algorithm follows from this single insight.